# RPS tracking

This notebook shows the full RPS-tracking loop: pull a recording span, examine it, run a neural predictor from the zoo, and run the blind DSP tracking ladder on the same audio. Each step is one or two calls into the library (`plots.explore`, `plots.dwym`, `zoo`, `tracking`).

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
for p in (ROOT, ROOT / "src"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import numpy as np
import tdseries as td

from plots import dwym, explore

## 1 · Datasets

`explore.datasets()` lists every published dataset pinned in `dload.lock`. Pass `sizes=True` for sample counts (fetches manifests).

In [ ]:
explore.datasets()

## 2 · Pull a recording span

The data cells stream from R2. This guard fails early, with a clear message, when credentials are missing.

In [ ]:
import os

import data_processing.streams  # noqa: F401 — loads the .env credentials

if not os.environ.get("AWS_ACCESS_KEY_ID"):
    raise RuntimeError(
        "No R2 credentials. The cells below stream dload datasets. "
        "Fill .env at the repo root (see docs/data-and-artifacts.md), then rerun."
    )

In [ ]:
rec = explore.pick("DREGON-frames", "free-flight_nosource_room1")
clip = rec.time[20.0:28.0].shift(-20.0)  # 8 s of stable flight, re-based to t=0
clip

`dwym` selects the figure from the entry names: here spectrogram + RPS rows + an audio player. `explore.pick` already coerced the raw entry names (`motors_command` → `rps`).

In [ ]:
dwym(clip, fmax=1500)

## 3 · A neural predictor from the zoo

`zoo.checkpoints` lists trained experiments from the R2 artifact store (cached locally). `zoo.load` returns the model as one Frame → Frame callable.

In [ ]:
import zoo

[row["experiment"] for row in zoo.checkpoints(task="rps_prediction")][:12]

In [ ]:
from data_processing.frames import resample_audio_series
from utils.audio import first_channel

audio16 = resample_audio_series(first_channel(clip["audio"]), 16000)
predictor = zoo.load("rps_simple_conv_v2_v4")
pred = predictor(td.Frame({"mixture": audio16}))

Overlay the prediction on the telemetry. `dwym` PIT-aligns `rps_pred` to `rps` before it draws.

In [ ]:
overlay = td.Frame({"audio": audio16, "rps": clip["rps"], "rps_pred": pred["rps_pred"]})
dwym(overlay, fmax=800)

## 4 · The blind DSP tracking ladder

Tracking stages are `td.Frame → td.Frame` callables (`tracking.stages`). `blind_seed_stage` finds the base speeds; the guarded VK stage captures and refines the trajectories. For the full calibrated annotation ladder, use `tracking.vit2dsp_stage`.

In [ ]:
import tracking as trk

tf = td.Frame({"audio": audio16, "rps_meas": clip["rps"], "meta": td.Frame({})})
ladder = trk.pipeline(trk.blind_seed_stage(4), trk.guarded(trk.vk_stage(trk.CAPTURE_CFG)))
tracked = ladder(tf)  # roughly a minute on CPU for 8 s of audio

Overlay the tracked trajectories on the telemetry the same way.

In [ ]:
overlay = td.Frame({"audio": audio16, "rps": clip["rps"], "rps_pred": tracked["rps"]})
dwym(overlay, fmax=800)

## 5 · The tracking log

Every stage appends one diagnostics dict to `meta["tracking"]`.

In [ ]:
for entry in tracked["meta"]["tracking"]:
    info = {k: v for k, v in entry.items() if k != "stage"}
    print(f"{entry['stage']:12s} {info}")

Next steps: `docs/notebook-primitives-tutorial.md` walks each primitive, `src/tracking/AGENTS.md` documents the stage API, and `notebooks/AGENTS.md` maps the other notebooks.